In [1]:
%load_ext autoreload
%autoreload 2

# Tabel 012 kvantorid 3

Lisaandmetena kasutatakse skripriga 910 kokku kogutud lemma pos korpuses esinemise statistikat.

Tasakaalus korpusest kogutakse kokku tipud - ülemus + vahetu alluv, kus:
* ülemuse sõnaliik on `S` ja kääne üks nendest: (term), (es), (kom), (abes)
* alluv eelneb lauses ülemusele;
* alluva sünrel on `nmod`, sõnaliik on `S` ja kääne (gen)

**Ülesande originaalpüstitus**

1. ülemuse sõnaliik = S ja kääne = ter, es, kom või ab + alluva sünrel = nmod, ja kääne = g
   
Tulemuste tabelis võiksid olla järgmised veerud: alluva lemma, alluva kääne, alluva arv, ülemuse lemma, ülemuse kääne, ülemuse arv, kogu lause, ?päringule vastav fragment, alluva lemma koguarv korpuses.

**Tulemus**

Tulmuseks on tabel sqlite formaadis.


Tabeli veerud
||||
|---|---|---|
|**child_lemma**| alluva lemma |---|
|**child_case**| alluva kääne |---|
|**child_number**| alluva arv |---|
|**parent_lemma**| ülemuse lemma |---|
|**parent_case**| ülemuse käänel |---|
|**parent_number**| ülemuse arv |---|
|**text**| ?päringule vastav fragment |---|
|**sentence**| teve lause tekst, kus ülemus ja alluv toodetud esile alakriipsudega  \_\_sõne\_\_ |---|
|**sentence_id**| lause id koondkorpuse andmebaasis|---|
|**child_lemma_total**| lemma + POS esinemise arv Tasakaalus korpuses |---|


In [2]:
import pandas as pd
from datetime import datetime

from data_helpers.syntax_graph import SyntaxGraph
from data_helpers.tasak_reader import TasakReader

# functions for creating database and collecting collocations
from collect_functions_012_quantifier_3 import *

In [3]:
# loeme sisse lemmade statistika ja teeme vastava dict
df_lemmas = pd.read_csv('lists/tasak_lemmas.tsv', sep='\t', keep_default_na=False)
lemmas_stat = { '%s\t%s' % (row['lemma'], row['POS'],): int(row['total']) for index, row in df_lemmas.iterrows()}
lemmas_stat['olema\tV']

633787

In [4]:
%%time

file_name = 'data/tasak.vert'

my_reader = TasakReader(
   file_name = file_name
)


CPU times: user 27 μs, sys: 18 μs, total: 45 μs
Wall time: 46.7 μs


In [5]:
%%time

TYPE = 'quantifier_3'
TABLENAME = f'{TYPE}'
BATCH_SIZE = 100000

date_time = datetime.now().strftime("%Y%m%d-%H%M%S")
db_file_name = f"tasak_{TYPE}_{date_time}.sqlite"

my_sqlite_db = DbMethods(db_file_name=db_file_name, table1_name=TYPE, table2_name=TYPE+'_examples')
my_sqlite_db.prep_coll_db()


# kollokatsioonid, tühjendatakse peale igat salvestamist
collocations = []
count = 0
for collection_id, graph in my_reader.get_sentences():
    
    count += 1
    if not collection_id:
        collection_id = count
    
    collocations = extract_something(graph, collection_id, collocations, lemmas_stat )


    if not collection_id == 0 and not count % BATCH_SIZE:
        my_sqlite_db.save_coll_to_db(collocations, collection_id)
        collocations = []
        
   
# saving last batch
my_sqlite_db.save_coll_to_db(collocations, collection_id)

#my_sqlite_db.index_fields()

data/tasak.vert


TSV lines:   9%|▉         | 1799798/20058039 [00:06<01:07, 269395.56it/s]

andmebaasi salvestatud kollokatsioonid kollektsioonidest: 0 - 346546


TSV lines:  18%|█▊        | 3563969/20058039 [00:12<01:02, 262401.89it/s]

andmebaasi salvestatud kollokatsioonid kollektsioonidest: 0 - 4759049


...

andmebaasi salvestatud kollokatsioonid kollektsioonidest: 0 - 14295795


TSV lines:  99%|█████████▉| 19906963/20058039 [01:11<00:00, 265486.97it/s]

andmebaasi salvestatud kollokatsioonid kollektsioonidest: 0 - 18956465


TSV lines: 100%|██████████| 20058039/20058039 [01:12<00:00, 276557.23it/s]

andmebaasi salvestatud kollokatsioonid kollektsioonidest: 0 - 18969731
CPU times: user 1min 12s, sys: 1.08 s, total: 1min 13s
Wall time: 1min 13s
